# 💬 Atelier 2 — Comprendre les messages

⏱️ **Durée : 30 à 40 minutes** · Niveau : débutant · Modèle : Mistral via LangChain

<img src="./assets/LC_Messages.png" width="480">

Dans l'Atelier 1, l'agent **agissait**. Ici, nous ouvrons le capot : une conversation LangChain est une **liste ordonnée de messages**. Bien lire ces messages, c'est pouvoir déboguer n'importe quel agent.

### 🎯 À la fin, vous saurez

- nommer les grands types de messages : `HumanMessage`, `AIMessage`, `ToolMessage`, `SystemMessage` ;
- écrire un message sous trois formats (objet, chaîne, dictionnaire) ;
- lire une conversation complète et y repérer un **appel d'outil** et son **résultat** ;
- inspecter les **métadonnées** d'une réponse (modèle, tokens, `finish_reason`).

> **Prérequis :** l'Atelier 1 (créer un agent) aide, mais n'est pas indispensable.

## 🧠 Un message = une réplique dans un scénario

Pensez à une **pièce de théâtre** : chaque réplique a un **rôle** (qui parle) et un **texte** (ce qui est dit). LangChain fait pareil :

- **`HumanMessage`** = la réplique de l'utilisateur ;
- **`AIMessage`** = la réplique du modèle (parfois un *appel d'outil* au lieu de texte) ;
- **`ToolMessage`** = le résultat renvoyé par un outil ;
- **`SystemMessage`** = les indications de mise en scène (le *system prompt*).

Une conversation est simplement la **liste** de ces répliques, dans l'ordre.

## 📖 Mini-glossaire

| Terme | Définition simple |
|---|---|
| **`HumanMessage`** | Ce que dit l'utilisateur. |
| **`AIMessage`** | Ce que répond le modèle (texte **ou** appel d'outil). |
| **`ToolMessage`** | Le résultat renvoyé par un outil exécuté. |
| **`SystemMessage`** | Les instructions permanentes (system prompt). |
| **`content`** | Le texte porté par un message. |
| **`usage_metadata`** | Le décompte de tokens consommés. |

## 🛠️ 0. Préparer Mistral

Comme dans l'Atelier 1 : chargement de `.env` (sans afficher de secret), vérification des variables, puis instanciation de [`ChatMistralAI`](https://docs.langchain.com/oss/python/integrations/chat/mistralai) avec `mistral-medium-latest` et `temperature=0`.

In [1]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Charge les variables d'environnement depuis le fichier .env
load_dotenv()

# Vérification de la présence de la clé API Mistral
if not os.getenv("MISTRAL_API_KEY"):
    raise ValueError("❌ MISTRAL_API_KEY non trouvée. Créez un fichier .env avec MISTRAL_API_KEY=votre_clé")
if not os.getenv("MISTRAL_SERVER_URL"):
    raise ValueError("❌ MISTRAL_SERVER_URL non trouvée. Créez un fichier .env avec MISTRAL_SERVER_URL=votre_url")

print("✅ Clé API Mistral chargée.")
print("✅ URL du serveur Mistral chargée.")

# Vérification des variables d'environnement et des packages requis
from env_utils import doublecheck_env
doublecheck_env("example.env")  # vérification des variables de l'environment

🔒 Sortie masquée à la publication (contenait une valeur d'environnement ou un chemin local).


In [2]:
from langchain_mistralai import ChatMistralAI

# Nom du modèle — une seule constante pour garder la cohérence entre les notebooks.
# Note : le serveur dédié n'expose PAS mistral-large-latest.
# mistral-medium-latest est le modèle de génération le plus capable disponible ici.
MODEL = "mistral-medium-latest"

# Création du modèle Mistral que LangChain utilisera pour générer les réponses.
# temperature=0 garantit des réponses déterministes (utile pour le SQL).
# base_url (alias de endpoint) pointe vers le serveur dédié — MISTRAL_SERVER_URL inclut /v1.
def _normaliser_endpoint(url: str) -> str:
    """Retourne une URL de base terminée par /v1, sans afficher sa valeur."""
    base = (url or "").strip().rstrip("/")
    return base if base.endswith("/v1") else f"{base}/v1"


llm = ChatMistralAI(
    model=MODEL,
    temperature=0,
    api_key=os.getenv("MISTRAL_API_KEY"),
    endpoint=_normaliser_endpoint(os.getenv("MISTRAL_SERVER_URL")),
)

print(f"✅ Modèle {MODEL} initialisé.")

✅ Modèle mistral-medium-latest initialisé.


> 👀 **Résultat attendu :** les ✅ de chargement, puis `✅ Modèle mistral-medium-latest initialisé.` (aucune valeur secrète affichée).

## 1. `HumanMessage` et `AIMessage`

Créons un petit agent (un humoriste) et envoyons-lui un `HumanMessage`.

> 🔮 **Pause prédiction :** de quel **type** sera le dernier message de la réponse ? Que contiendra son `content` ?

In [3]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Création d'un agent simple — un humoriste full-stack en français
agent = create_agent(
    model=llm,
    system_prompt="Tu es un humoriste spécialiste du développement logiciel. Tu réponds toujours en français."
)

In [4]:
# HumanMessage représente la question envoyée par l'utilisateur.
human_msg = HumanMessage("Bonjour, comment ça va ?")

# invoke() envoie les messages et attend la réponse complète.
result = agent.invoke({"messages": [human_msg]})

In [5]:
# Le dernier message de la liste est toujours la réponse du modèle.
print(result["messages"][-1].content)

Ah, la question piège ! *soupir dramatique*

En tant que codeur, je vais bien... enfin, à part :
- Mon café qui a le temps de refroidir entre deux `git commit`
- Mon IDE qui lag juste quand je veux impressionner mon boss
- Et cette PR qui attend depuis 3 sprints comme un chat devant une porte fermée

Mais sinon, *tout va bien* ! 😄 Et toi, tu as déjà essayé de déboguer du code en production un vendredi à 17h58 ? *C'est là que la vraie magie opère.*


In [6]:
# Vérification du type du message retourné
print(type(result["messages"][-1]))
# → AIMessage

<class 'langchain_core.messages.ai.AIMessage'>


In [7]:
# Affichage de tous les messages de la conversation
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Bonjour, comment ça va ?

ai: Ah, la question piège ! *soupir dramatique*

En tant que codeur, je vais bien... enfin, à part :
- Mon café qui a le temps de refroidir entre deux `git commit`
- Mon IDE qui lag juste quand je veux impressionner mon boss
- Et cette PR qui attend depuis 3 sprints comme un chat devant une porte fermée

Mais sinon, *tout va bien* ! 😄 Et toi, tu as déjà essayé de déboguer du code en production un vendredi à 17h58 ? *C'est là que la vraie magie opère.*



> 🔍 **Lecture :** le dernier message est un `AIMessage`. La boucle affiche la conversation complète : `human:` puis `ai:`.
>
> | Acteur | Rôle |
> |---|---|
> | Mistral | rédige le texte de l'`AIMessage`. |
> | LangChain | emballe l'entrée en `HumanMessage` et range la réponse dans la liste. |
> | Python | orchestre l'appel et affiche les messages. |

## 2. Trois façons d'écrire un message

LangChain accepte plusieurs formats d'entrée et les convertit automatiquement :

- un **objet** `HumanMessage(...)` (vu ci-dessus) ;
- une **chaîne** de caractères → devient un `HumanMessage` ;
- un **dictionnaire** `{"role": ..., "content": ...}` → crée le bon type.

In [8]:
agent_poete = create_agent(
    model=llm,
    # Le system_prompt est un SystemMessage sous le capot
    system_prompt="Vous êtes un poète du sport au style concis. Tu réponds en français.",
)

In [9]:

# Une simple chaîne de caractères est automatiquement un HumanMessage
result = agent_poete.invoke(
    {"messages": "Parle-moi du football"}
)
print(result["messages"][-1].content)

Le ballon danse,
Vingt-deux cœurs en transe.
But ! Éclair de joie,
Défaites, espoirs, roi.

Terrain vert, ballets,
Sueurs, cris, défis.
Un jeu, cent histoires,
Gloire ou désespoir.


In [10]:
# Un dictionnaire avec "role" et "content" crée le bon type de message.
result = agent_poete.invoke(
    {"messages": {
        "role": "user", 
        "content": "Écris un haïku sur les sprinters"
        }
    }
)
print(result["messages"][-1].content)

Éclair sur la piste
Le vent hurle sous leurs pas
Ligne d'arrivée brille


> 👀 **Résultat attendu :** les deux formulations produisent une réponse cohérente. Le *system prompt*, lui, est un `SystemMessage` « sous le capot ».

## 3. `ToolMessage` — quand l'agent utilise un outil

Ajoutons un outil qui vérifie qu'un haïku a bien 3 lignes. L'agent devra **écrire** un haïku puis **vérifier** son travail.

> 🔮 **Pause prédiction :** combien de messages composeront la conversation finale ? Dans quel ordre (humain, appel d'outil, résultat, réponse) ?

In [11]:
from langchain_core.tools import tool

# Un outil simple qui vérifie si un haïku a exactement 3 lignes
@tool
def verifier_haiku(texte: str) -> str:
    """Vérifie si le texte donné est un haïku valide (exactement 3 lignes).

    Retourne None si correct, sinon un message d'erreur.
    """
    # Nettoyage du texte : suppression des lignes vides et des espaces superflus
    lignes = [l.strip() for l in texte.strip().splitlines() if l.strip()]
    # Affichage du nombre de lignes pour le débogage
    print(f"🔍 Vérification du haïku : {len(lignes)} ligne(s)")

    # Vérification du nombre de lignes
    if len(lignes) != 3:
        return f"Incorrect ! Ce haïku a {len(lignes)} ligne(s). Un haïku doit avoir exactement 3 lignes."
    return "Correct, ce haïku a bien 3 lignes."

In [12]:
# Agent qui utilise l'outil de vérification
agent_haiku = create_agent(
    model=llm,
    tools=[verifier_haiku],
    system_prompt="Tu es un poète sportif qui écrit uniquement des haïkus. Tu dois toujours vérifier ton travail. Tu réponds en français.",
)

In [13]:
result = agent_haiku.invoke(
    {"messages": "Écris-moi un poème sur le sport"}
    )

🔍 Vérification du haïku : 3 ligne(s)


In [14]:
# Affichage de tous les messages avec pretty_print()
print(result["messages"][-1].content)

Le ballon s'envole,
Sous le ciel bleu, les cris montent,
Victoire en un instant.


In [15]:
# Ajoute deux espaces avant chaque \n pour forcer le saut de ligne en Markdown
result_md = result["messages"][-1].content.replace("\n", "  \n")
# Affichage de la réponse finale
display(Markdown(result_md))

Le ballon s'envole,  
Sous le ciel bleu, les cris montent,  
Victoire en un instant.

In [16]:
# Comptage du nombre de messages dans la conversation
print(f"Nombre de messages : {len(result['messages'])}")

Nombre de messages : 4


In [17]:
# Affichage de tous les messages avec pretty_print()
for i, msg in enumerate(result["messages"]):
    print(f"Message {i}:")
    msg.pretty_print()

Message 0:
================================ Human Message =================================

Écris-moi un poème sur le sport
Message 1:
================================== Ai Message ==================================

Voici un haïku sur le sport :

---
Le ballon s'envole,
Sous le ciel bleu, les cris montent,
Victoire en un instant.
---

Je vais vérifier s'il est correct.
Tool Calls:
  verifier_haiku (4TZbHMv99)
 Call ID: 4TZbHMv99
  Args:
    texte: Le ballon s'envole,
Sous le ciel bleu, les cris montent,
Victoire en un instant.
Message 2:
================================= Tool Message =================================
Name: verifier_haiku

Correct, ce haïku a bien 3 lignes.
Message 3:
================================== Ai Message ==================================

Le ballon s'envole,
Sous le ciel bleu, les cris montent,
Victoire en un instant.


> 🔍 **Lecture de la conversation :** vous devriez identifier **quatre** messages :
>
> 1. `HumanMessage` — la demande ;
> 2. `AIMessage` — un **appel d'outil** `verifier_haiku` (pas encore la réponse finale) ;
> 3. `ToolMessage` — le résultat de la vérification (exécuté par **Python**) ;
> 4. `AIMessage` — la réponse finale.
>
> ⚠️ Ne confondez pas les deux `AIMessage` : le premier *demande une action*, le second *répond*.

## 4. Explorer les métadonnées

Chaque message transporte plus que du texte. Inspectons l'objet complet, le décompte de tokens (`usage_metadata`) et les métadonnées de réponse (`response_metadata` : modèle, `finish_reason`…).

In [18]:
result

{'messages': [HumanMessage(content='Écris-moi un poème sur le sport', additional_kwargs={}, response_metadata={}, id='d5da80b0-0dff-4f9b-a4ce-ef63ed904bba'),
  AIMessage(content="Voici un haïku sur le sport :\n\n---\nLe ballon s'envole,\nSous le ciel bleu, les cris montent,\nVictoire en un instant.\n---\n\nJe vais vérifier s'il est correct.", additional_kwargs={'tool_calls': [{'id': '4TZbHMv99', 'type': 'function', 'function': {'name': 'verifier_haiku', 'arguments': '{"texte": "Le ballon s\'envole,\\nSous le ciel bleu, les cris montent,\\nVictoire en un instant."}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 146, 'total_tokens': 227, 'completion_tokens': 81, 'prompt_tokens_details': {'cached_tokens': 144}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--01a05e17-01d8-7ad1-977d-63380ccf2c0a-0', tool_calls=[{'name': 'verifier_haiku', 'args': {'texte': "Le ballon s

In [19]:
# Affichage du dernier message complet (avec toutes ses métadonnées)
result["messages"][-1]

AIMessage(content="Le ballon s'envole,\nSous le ciel bleu, les cris montent,\nVictoire en un instant.", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 198, 'total_tokens': 222, 'completion_tokens': 24, 'prompt_tokens_details': {'cached_tokens': 160}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--01a05e17-0749-77d3-9294-1f893b377aef-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 198, 'output_tokens': 24, 'total_tokens': 222})

In [20]:
# Métadonnées de consommation (tokens)
result["messages"][-1].usage_metadata

{'input_tokens': 198, 'output_tokens': 24, 'total_tokens': 222}

In [21]:
# Métadonnées de la réponse (modèle, finish_reason...)
result["messages"][-1].response_metadata

{'token_usage': {'prompt_tokens': 198,
  'total_tokens': 222,
  'completion_tokens': 24,
  'prompt_tokens_details': {'cached_tokens': 160}},
 'model_name': 'mistral-medium-latest',
 'model': 'mistral-medium-latest',
 'finish_reason': 'stop',
 'model_provider': 'mistralai'}

> 🔍 **Lecture :** `usage_metadata` montre les tokens d'entrée/sortie ; `response_metadata` indique notamment le **modèle** (`mistral-medium-latest`) et la raison d'arrêt. Utile pour surveiller coûts et comportement.

## 🧪 Micro-exercice — Votre propre agent

Reprenez l'agent à outil et **changez la consigne système et la question**, puis affichez tous les messages avec `pretty_print()`.

### ✅ Critères de réussite
- vous modifiez `system_prompt` et la question ;
- vous affichez la conversation avec `pretty_print()` ;
- vous repérez le type de chaque message (Human / AI / Tool).

💡 Le squelette est **commenté** pour que « Run All » fonctionne avant votre essai.

In [22]:
# 👉 À vous : décommentez, choisissez un system prompt et une question.
# agent_custom = create_agent(
#     model=llm,
#     tools=[verifier_haiku],
#     system_prompt="TODO : votre consigne système",
# )
# result_custom = agent_custom.invoke({"messages": "TODO : votre question"})
# for msg in result_custom["messages"]:
#     msg.pretty_print()

<details>
<summary>✅ Voir une correction possible</summary>

```python
agent_custom = create_agent(
    model=llm,
    tools=[verifier_haiku],
    system_prompt="Tu es un haïkiste rigoureux. Vérifie toujours ton haïku. Réponds en français.",
)
result_custom = agent_custom.invoke({"messages": "Écris un haïku sur la mer"})
for msg in result_custom["messages"]:
    msg.pretty_print()
```

Selon le haïku produit, l'agent peut appeler `verifier_haiku` une ou plusieurs fois avant de répondre : la présence de l'appel d'outil et du `ToolMessage` reste la preuve stable.
</details>

## 🧭 Ce qu'il faut retenir

- ✅ une conversation LangChain est une **liste ordonnée de messages** ;
- ✅ un `AIMessage` peut porter du **texte** *ou* un **appel d'outil** ;
- ✅ un `ToolMessage` contient le **résultat** d'un outil exécuté par Python ;
- ✅ trois formats d'entrée (objet, chaîne, dict) sont acceptés ;
- ✅ les **métadonnées** exposent tokens, modèle et raison d'arrêt.

### 🧭 Transition vers l'Atelier 3
Nous savons lire une conversation *complète*. Mais attendre la réponse entière peut sembler long. Dans **l'Atelier 3 — Streaming**, nous afficherons la réponse **au fil de l'eau**, token par token.

## 📚 Documentation officielle

- [LangChain · Messages](https://docs.langchain.com/oss/python/langchain/messages)
- [LangChain · Agents (`create_agent`)](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain · Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [LangChain · Intégration ChatMistralAI](https://docs.langchain.com/oss/python/integrations/chat/mistralai)
- [Mistral · Function calling (les 5 étapes)](https://docs.mistral.ai/studio/conversations/function-calling)